# 51 Visualization

High-quality plotly charts from the Stage 4 analysis.

**Prerequisite:** Run `41_analysis.ipynb` first.

**Outputs (HTML + PNG):** `documentation/visualizations/`
- `pagerank_top20` — top 20 persons by page-rank
- `gender_by_occupation` — gender % by occupation (unique persons)
- `gender_by_occurrence` — gender % weighted by appearance count
- `age_distribution` — age at first appearance histogram
- `age_by_occupation` — age box plots per occupation
- `appearance_count_distribution` — how many episodes each guest appears in

In [ ]:
from pathlib import Path
from datetime import UTC, datetime, timezone
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    # Notebook may be opened from the notebooks folder.
    for parent in [repo_root] + list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter
from process.io_guardrails import atomic_write_text

VIZ_DIR = Path('documentation/visualizations')
VIZ_DIR.mkdir(parents=True, exist_ok=True)

catalogue = pd.read_csv('data/40_analysis/guest_catalogue.csv', dtype=str).fillna('')
catalogue['cluster_size'] = catalogue['cluster_size'].astype(int)
catalogue['age_at_first_appearance'] = pd.to_numeric(catalogue['age_at_first_appearance'], errors='coerce')
catalogue['episode_count'] = pd.to_numeric(catalogue['episode_count'], errors='coerce')

pr_df = pd.read_csv('data/40_analysis/pagerank_persons.csv', dtype=str)
pr_df['pagerank'] = pr_df['pagerank'].astype(float)
top20_pr = pr_df.head(20).sort_values('pagerank')

print('Loaded:', len(catalogue), 'Wikidata-matched persons,', len(pr_df), 'page-rank entries')


## Page-rank: top 20 persons

In [ ]:
fig_pr = go.Figure(
    go.Bar(
        x=top20_pr['pagerank'],
        y=top20_pr['label_de'],
        orientation='h',
        marker=dict(color=top20_pr['pagerank'], colorscale='Blues', showscale=True,
                    colorbar=dict(title='Page-rank')),
        text=[f'{v:.5f}' for v in top20_pr['pagerank']],
        textposition='outside',
    )
)
fig_pr.update_layout(
    title={'text': 'Top 20 persons by page-rank<br>'
                   '<sub>Wikidata graph · P31/P279 taxonomy edges excluded · alpha=0.85</sub>', 'x': 0.5},
    xaxis_title='Page-rank score',
    height=600, width=900, margin=dict(l=200, r=120), font=dict(size=12),
)
fig_pr.write_html(str(VIZ_DIR / 'pagerank_top20.html'))
fig_pr.write_image(str(VIZ_DIR / 'pagerank_top20.png'), scale=2)
fig_pr.show()


## Gender distribution by occupation

In [ ]:
exploded = catalogue.copy()
exploded['occupation'] = exploded['occupations'].str.split('|')
exploded = exploded.explode('occupation')
exploded = exploded[exploded['occupation'].str.strip() != '']

occ_freq = exploded['occupation'].value_counts()
TOP_OCC = occ_freq[occ_freq >= 20].index.tolist()
print(f'Occupations with >= 20 persons: {len(TOP_OCC)}')

piv_persons = (
    exploded[exploded['occupation'].isin(TOP_OCC)]
    .groupby(['occupation', 'gender'])['wikidata_id'].count()
    .unstack(fill_value=0).reindex(columns=['male', 'female', ''], fill_value=0)
)
piv_persons.columns = ['Male', 'Female', 'Unknown']
piv_persons_pct = piv_persons.div(piv_persons.sum(axis=1), axis=0) * 100

piv_app = (
    exploded[exploded['occupation'].isin(TOP_OCC)]
    .groupby(['occupation', 'gender'])['cluster_size'].sum()
    .unstack(fill_value=0).reindex(columns=['male', 'female', ''], fill_value=0)
)
piv_app.columns = ['Male', 'Female', 'Unknown']
piv_app_pct = piv_app.div(piv_app.sum(axis=1), axis=0) * 100

sort_order = piv_persons_pct['Female'].sort_values(ascending=True).index.tolist()
piv_persons_pct = piv_persons_pct.loc[sort_order]
piv_app_pct = piv_app_pct.loc[sort_order]
COLORS = {'Male': '#4878cf', 'Female': '#e8534a', 'Unknown': '#cccccc'}


In [ ]:
def make_stacked_bar(pct_df, totals, title):
    fig = go.Figure()
    for gender in ['Male', 'Female', 'Unknown']:
        if gender not in pct_df.columns:
            continue
        vals = pct_df[gender].values
        fig.add_trace(go.Bar(
            name=gender, y=pct_df.index.tolist(), x=vals, orientation='h',
            marker_color=COLORS[gender],
            text=[f'{v:.0f}% (n={t})' if v >= 5 else '' for v, t in zip(vals, totals)],
            textposition='inside', textfont=dict(color='white', size=10),
        ))
    fig.update_layout(
        barmode='stack', title={'text': title, 'x': 0.5},
        xaxis=dict(title='Percentage', range=[0, 100]),
        height=max(400, len(pct_df) * 35 + 120), width=950,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        shapes=[dict(type='line', x0=50, x1=50, y0=-0.5, y1=len(pct_df) - 0.5,
                     line=dict(color='black', dash='dash', width=1))],
        margin=dict(l=170, r=30), font=dict(size=12),
    )
    return fig

totals_persons = piv_persons.sum(axis=1).loc[sort_order].values
totals_app = piv_app.sum(axis=1).loc[sort_order].values

fig_ind = make_stacked_bar(piv_persons_pct, totals_persons,
    'Gender by occupation — unique persons<br>'
    '<sub>n = count of Wikidata-matched canonical entities · ordered by female %</sub>')
fig_ind.write_html(str(VIZ_DIR / 'gender_by_occupation.html'))
fig_ind.write_image(str(VIZ_DIR / 'gender_by_occupation.png'), scale=2)
fig_ind.show()

fig_app = make_stacked_bar(piv_app_pct, totals_app,
    'Gender by occupation — weighted by appearances<br>'
    '<sub>n = total guest slots filled · same ordering as above</sub>')
fig_app.write_html(str(VIZ_DIR / 'gender_by_occurrence.html'))
fig_app.write_image(str(VIZ_DIR / 'gender_by_occurrence.png'), scale=2)
fig_app.show()


## Age at first appearance

In [ ]:
aged = catalogue[catalogue['age_at_first_appearance'].notna()].copy()

fig_age = go.Figure()
for gender, color in [('male', '#4878cf'), ('female', '#e8534a')]:
    subset = aged[aged['gender'] == gender]['age_at_first_appearance']
    fig_age.add_trace(go.Histogram(
        x=subset, name=gender.capitalize(), opacity=0.75, nbinsx=20, marker_color=color,
    ))
fig_age.update_layout(
    barmode='overlay',
    title={'text': f'Age at first appearance on Markus Lanz<br>'
                   f'<sub>{len(aged)} persons · mean ≈ {aged["age_at_first_appearance"].mean():.0f} years · Wikidata P569</sub>',
           'x': 0.5},
    xaxis_title='Age at first appearance (years)', yaxis_title='Number of persons',
    height=450, width=850, legend=dict(orientation='h', yanchor='bottom', y=1.02), font=dict(size=12),
)
fig_age.write_html(str(VIZ_DIR / 'age_distribution.html'))
fig_age.write_image(str(VIZ_DIR / 'age_distribution.png'), scale=2)
fig_age.show()


In [ ]:
aged_occ = aged[aged['occupations'].str.strip() != ''].copy()
aged_occ['occupation'] = aged_occ['occupations'].str.split('|')
aged_occ = aged_occ.explode('occupation')
aged_occ = aged_occ[aged_occ['occupation'].isin(TOP_OCC)]

occ_order = (aged_occ.groupby('occupation')['age_at_first_appearance']
             .median().sort_values().index.tolist())

fig_age_occ = go.Figure()
for occ in occ_order:
    data = aged_occ[aged_occ['occupation'] == occ]['age_at_first_appearance']
    fig_age_occ.add_trace(go.Box(y=data, name=occ, boxmean=True,
                                  marker_color='#5e81ac', line_color='#4c566a'))
fig_age_occ.update_layout(
    title={'text': 'Age at first appearance by occupation<br>'
                   '<sub>Box = IQR · dot = mean · ordered by median age</sub>', 'x': 0.5},
    yaxis_title='Age at first appearance (years)',
    height=520, width=1050, showlegend=False, font=dict(size=11),
    margin=dict(b=120),
)
fig_age_occ.update_xaxes(tickangle=-35)
fig_age_occ.write_html(str(VIZ_DIR / 'age_by_occupation.html'))
fig_age_occ.write_image(str(VIZ_DIR / 'age_by_occupation.png'), scale=2)
fig_age_occ.show()


## Episode appearance count

In [ ]:
ep_counts = catalogue[catalogue['episode_count'].notna()].copy()
fig_ep = px.histogram(
    ep_counts, x='episode_count', color='gender', nbins=30, barmode='overlay', opacity=0.75,
    color_discrete_map={'male': '#4878cf', 'female': '#e8534a', '': '#cccccc'},
    title=f'How many episodes does each guest appear in?<br>'
          f'<sub>{len(ep_counts)} Wikidata-matched persons · median = {ep_counts["episode_count"].median():.0f} · max = {int(ep_counts["episode_count"].max())}</sub>',
    labels={'episode_count': 'Number of episode appearances', 'gender': 'Gender'},
)
fig_ep.update_layout(yaxis_title='Number of persons', height=450, width=850, font=dict(size=12))
fig_ep.write_html(str(VIZ_DIR / 'appearance_count_distribution.html'))
fig_ep.write_image(str(VIZ_DIR / 'appearance_count_distribution.png'), scale=2)
fig_ep.show()

print()
print('All charts written to:', str(VIZ_DIR.resolve()))
